## 🔍 设备信息查询与检查

这些方法用于在代码运行前确认硬件环境和状态：
- torch.cuda.is_available()：检查当前系统是否安装了支持 CUDA 的 NVIDIA 显卡，以及 PyTorch 是否已编译为支持 CUDA。这是写代码时最常用的判断逻辑。
- torch.cuda.device_count()：返回当前机器上可用的 GPU 数量。
- torch.cuda.get_device_name(device=None)：获取指定 GPU 的名称（如 "NVIDIA GeForce RTX 3090"）。
- torch.cuda.get_device_capability(device=None)：获取指定 GPU 的 CUDA 计算能力（例如返回元组 (7, 5)），常用于判断显卡是否支持某些高级特性。
- torch.version.cuda：查看当前 PyTorch 编译时使用的 CUDA 版本（如 "11.8"）。

In [ ]:
import torch

print("------ PyTorch CUDA 环境侦察报告 ------")

# 1. torch.cuda.is_available()
# 这是最基础的“通行证”。如果返回 False，说明要么没装 NVIDIA 显卡，要么 PyTorch 装的是 CPU 版本。
is_cuda = torch.cuda.is_available()
print(f"1. CUDA 是否可用: {is_cuda}")

if is_cuda:
    # 2. torch.cuda.device_count()
    # 看看你的机器是“单枪匹马”还是“多卡并行”。
    gpu_count = torch.cuda.device_count()
    print(f"2. 可用的 GPU 数量: {gpu_count}")

    # 3. torch.version.cuda
    # 查看你当前安装的 PyTorch 是基于哪个版本的 CUDA 编译的。
    # 注意：这不一定等于你系统驱动支持的最高 CUDA 版本。
    print(f"3. PyTorch 编译时的 CUDA 版本: {torch.version.cuda}")

    # 4. torch.cuda.get_device_name(device=None)
    # 打印显卡的具体型号。device 参数默认为 0（第一张显卡）。
    # 如果是多卡机器，可以通过传入索引（如 1, 2）来查看其他显卡。
    for i in range(gpu_count):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"4. GPU {i} 的名称: {gpu_name}")

    # 5. torch.cuda.get_device_capability(device=None)
    # 获取显卡的“计算能力”（Compute Capability），返回一个 (主版本号, 次版本号) 的元组。
    # 比如 (7, 5) 代表图灵架构（如 RTX 20系列），(8, 6) 代表安培架构（如 RTX 30系列）。
    # 这在判断显卡是否支持某些高级特性（如 bfloat16、FlashAttention）时非常有用。
    for i in range(gpu_count):
        capability = torch.cuda.get_device_capability(i)
        print(f"5. GPU {i} 的计算能力: {capability}")

else:
    print("⚠️ 当前环境没有检测到可用的 NVIDIA GPU，程序将默认在 CPU 上运行。")

## 💡 重点补充说明： 
1. torch.cuda.is_available() 的潜台词：
- <font color='red'>它不仅检查你有没有显卡，还检查 PyTorch 是否成功连接了显卡驱动。</font>如果你明明有 4090 显卡但它返回 False，通常是因为你安装的是 CPU 版本的 PyTorch，或者显卡驱动没装好。
2. 关于 get_device_capability (计算能力)：
- 这个元组非常关键。
  - 例如，如果你的计算能力是 (8, 0) 或更高（如 RTX 30/40系列、A100），你就可以放心使用 PyTorch 的最新高级优化特性（如 TF32 加速、FlashAttention 等）；
  - 如果是较老的 (6, 1)（如 GTX 1080 Ti），在运行某些新模型时可能会遇到兼容性问题。
3. torch.version.cuda 的误区：
- <font color='red'>这个版本号指的是PyTorch 官方打包时用的 CUDA 版本，而不是你电脑显卡驱动支持的最高版本</font>。比如你驱动支持 CUDA 12.4，但你装的 PyTorch 可能是基于 CUDA 11.8 编译的，这里就会显示 "11.8"。